In [1]:
from sklearn.datasets import make_regression
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

np.random.seed(42)

# Generate synthetic regression data
X, y = make_regression(n_samples=100, n_features=2, n_informative=2, 
                       n_targets=1, noise=50, random_state=42)

# Create DataFrame
df = pd.DataFrame({'feature1': X[:, 0], 'feature2': X[:, 1], 'target': y})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (100, 3)


,feature1,feature2,target
0,-1.191303,0.656554,-22.779796
1,0.058209,-1.142970,-107.569629
2,0.586857,2.190456,201.122932
3,0.473238,-0.072829,1.480178
4,0.738467,0.171368,111.798503


In [2]:
fig = px.scatter_3d(df, x='feature1', y='feature2', z='target',
                    title='Raw Data: 3D Scatter Plot',
                    labels={'feature1': 'Feature 1', 'feature2': 'Feature 2', 'target': 'Target'})
fig.update_traces(marker=dict(size=4, opacity=0.7))
fig.show()

In [3]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)

# Predict
y_pred = lr.predict(X_test)

# Evaluate
print("=== Model Evaluation ===")
print(f"MAE:  {mean_absolute_error(y_test, y_pred):.2f}")
print(f"MSE:  {mean_squared_error(y_test, y_pred):.2f}")
print(f"R²:   {r2_score(y_test, y_pred):.4f}")
print(f"\nCoefficients: {lr.coef_}")
print(f"Intercept: {lr.intercept_:.2f}")

=== Model Evaluation ===
MAE:  48.34
MSE:  3865.69
R²:   0.6996

Coefficients: [80.60169788 72.03894822]
Intercept: 0.33


In [4]:
# Create meshgrid for surface plot - use data range, not arbitrary -5 to 5
x_range = np.linspace(X[:, 0].min(), X[:, 0].max(), 50)
y_range = np.linspace(X[:, 1].min(), X[:, 1].max(), 50)
xGrid, yGrid = np.meshgrid(x_range, y_range)

# Flatten meshgrid to create input for prediction
grid_points = np.column_stack([xGrid.ravel(), yGrid.ravel()])

# Predict z values for all grid points
zGrid = lr.predict(grid_points)

# Reshape predictions back to meshgrid shape for surface plot
zGrid = zGrid.reshape(xGrid.shape)

print(f"Meshgrid shape: {xGrid.shape}")
print(f"Prediction surface shape: {zGrid.shape}")

Meshgrid shape: (50, 50)
Prediction surface shape: (50, 50)


In [5]:
# Create figure with both scatter and surface
fig = go.Figure()

# Add original data as scatter points
fig.add_trace(go.Scatter3d(
    x=df['feature1'], y=df['feature2'], z=df['target'],
    mode='markers',
    name='Actual Data',
    marker=dict(size=4, color=df['target'], colorscale='Viridis', opacity=0.7)
))

# Add regression plane as surface
fig.add_trace(go.Surface(
    x=xGrid, y=yGrid, z=zGrid,
    name='Prediction Surface',
    colorscale='Reds',
    opacity=0.5,
    showscale=False
))

# Layout customization
fig.update_layout(
    title='Linear Regression: Data + Prediction Surface',
    scene=dict(
        xaxis_title='Feature 1',
        yaxis_title='Feature 2', 
        zaxis_title='Target',
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
    ),
    width=900,
    height=700,
    legend=dict(x=0.02, y=0.98)
)

fig.show()

In [6]:
# Calculate residuals for test set
residuals = y_test - y_pred

# Create residuals DataFrame
residual_df = pd.DataFrame({
    'feature1': X_test[:, 0],
    'feature2': X_test[:, 1], 
    'residual': residuals
})

# Plot residuals in 3D
fig = px.scatter_3d(residual_df, x='feature1', y='feature2', z='residual',
                    color='residual', color_continuous_scale='RdYlBu_r',
                    title='Residuals Plot (Ideal: scattered around z=0)',
                    labels={'residual': 'Residual (Actual - Predicted)'})

# Add zero plane for reference
fig.add_trace(go.Surface(
    x=[X_test[:, 0].min(), X_test[:, 0].max()],
    y=[X_test[:, 1].min(), X_test[:, 1].max()],
    z=[[0, 0], [0, 0]],
    opacity=0.2, showscale=False, colorscale='Greys'
))

fig.update_layout(coloraxis_colorbar=dict(title="Residual"))
fig.show()

# Residual statistics
print(f"\n=== Residual Analysis ===")
print(f"Mean residual: {residuals.mean():.4f} (should be ~0)")
print(f"Std residual: {residuals.std():.4f}")
print(f"Min residual: {residuals.min():.4f}")
print(f"Max residual: {residuals.max():.4f}")


=== Residual Analysis ===
Mean residual: 5.0294 (should be ~0)
Std residual: 61.9709
Min residual: -158.6019
Max residual: 114.0374
